# Prices: S&P 500 features, labels, and Fed rate

Downloads daily S&P 500 data from Yahoo Finance and the Fed's target rate from FRED, and combines them into one table with one row per trading day from Jan 3, 2022 to Sep 30, 2026.

- **log_return**: the log return of the S&P 500
- **hl_range**: how much the index moves within the day, as a fraction of the price
- **log_volume_change**: how much the volume changes from the previous day
- **label**: 1 if tomorrow's close is higher than today's, else 0
- **rate_upper**: upper bound of the Fed's target rate (FRED series DFEDTARU) in effect after any decision that day. Used to pick which Kalshi markets to read.

Results are saved to `prices.csv`.

In [54]:
# import dependencies
import numpy as np
import pandas as pd
import yfinance as yf

In [55]:
# download S&P 500 data from Yahoo Finance
raw = yf.download("^GSPC", start="2021-12-31", end="2026-10-02", progress=False, auto_adjust=False)
raw.columns = raw.columns.get_level_values(0)
print(raw.tail(3)) # check the last 3 rows of the raw data

Price         Adj Close        Close         High          Low         Open  \
Date                                                                          
2026-09-29  7670.839844  7670.839844  7699.600098  7653.549805  7699.600098   
2026-09-30  7651.540039  7651.540039  7722.879883  7651.540039  7688.990234   
2026-10-01  7666.450195  7666.450195  7684.750000  7616.779785  7666.470215   

Price           Volume  
Date                    
2026-09-29  5023590000  
2026-09-30  4276550000  
2026-10-01  5783800000  


Add log_return, high-low range, log_volume chnage, and label to data

In [56]:
df = pd.DataFrame(index=raw.index)

# get log returns
df["log_return"] = np.log(raw["Close"] / raw["Close"].shift(1))
#hl_range: how much the index moves within the day, as a fraction of the price. calm = small and volatile = big range.
df["hl_range"] = (raw["High"] - raw["Low"]) / raw["Close"] # come from same day, so no need to shift    
#log_volume_change: how much the volume changes from the previous day
df["log_volume_change"] = np.log(raw["Volume"] / raw["Volume"].shift(1))
# df["log_volume_change"] = np.log(raw["Volume"]).diff()
df["label"] = (raw["Close"].shift(-1) > raw["Close"]).astype(int) # label: 1 if tomorrow's close is higher than today's, else 0

df = df.loc["2022-01-03":"2026-09-30"] # trim to the first and last trading days of 2022-2026

/Users/kimmytran/APS360-LSTM/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Checks

In [57]:
print(len(df))
print(df.isna().sum())
print(df["label"].mean())
print(df.head(1)); print(df.tail(1))

1190
log_return           0
hl_range             0
log_volume_change    0
label                0
dtype: int64
0.5277310924369748
            log_return  hl_range  log_volume_change  label
Date                                                      
2022-01-03    0.006354   0.00802           0.358128      0
            log_return  hl_range  log_volume_change  label
Date                                                      
2026-09-30   -0.002519  0.009324          -0.160998      1


Add Fed rate data

In [58]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DFEDTARU"
fed_rate = pd.read_csv(url, index_col=[0], parse_dates=[0])
fed_rate = fed_rate.iloc[:, 0]
next_dates = df.index + pd.Timedelta(days=1) # every trading day shifted forward by 1 day
df["rate_upper"] = fed_rate.reindex(next_dates, method="ffill").values

Checks


In [59]:
# check
print(df.loc["2022-03-15":"2022-03-18", "rate_upper"])
print(df["rate_upper"].isna().sum())
print(df["rate_upper"][df["rate_upper"].diff() != 0])   # every date the rate changed

Date
2022-03-15    0.25
2022-03-16    0.50
2022-03-17    0.50
2022-03-18    0.50
Name: rate_upper, dtype: float64
0
Date
2022-01-03    0.25
2022-03-16    0.50
2022-05-04    1.00
2022-06-15    1.75
2022-07-27    2.50
2022-09-21    3.25
2022-11-02    4.00
2022-12-14    4.50
2023-02-01    4.75
2023-03-22    5.00
2023-05-03    5.25
2023-07-26    5.50
2024-09-18    5.00
2024-11-07    4.75
2024-12-18    4.50
2025-09-17    4.25
2025-10-29    4.00
2025-12-10    3.75
2026-09-16    4.00
Name: rate_upper, dtype: float64


# Saving data/files
Creates a new data folder if one doesn't exist yet. 

In [60]:
# creates data folder
from pathlib import Path
Path("data").mkdir(exist_ok=True)

Results are saved to `prices.csv` in data folder.

In [61]:
# write the dataframe to a CSV file
df.to_csv("data/prices.csv")

Check

In [62]:
# check if it reads correctly
check = pd.read_csv("data/prices.csv", index_col="Date", parse_dates=True)
print(check.shape)
print(check.head(2))


(1190, 5)
            log_return  hl_range  log_volume_change  label  rate_upper
Date                                                                  
2022-01-03    0.006354  0.008020           0.358128      0        0.25
2022-01-04   -0.000630  0.009252           0.200844      0        0.25
